In [1]:
import pandas as pd
import numpy as np

files = [
    "Cherkasy_commercial.csv", "Chernihiv_commercial.csv", "Chernivtsi_commercial.csv", "Dnipro_commercial.csv", "Ivano-Frankivsk_commercial.csv",
    "Kharkiv_commercial.csv", "Khmelnytskyi_commercial.csv", "Kropyvnytskyi_commercial.csv",
    "Lutsk_commercial.csv", "Lviv_commercial.csv", "Mykolaiv_commercial.csv", "Odesa_commercial.csv", "Poltava_commercial.csv", "Rivne_commercial.csv",
    "Sumy_commercial.csv", "Ternopil_commercial.csv", "Uzhhorod_commercial.csv", "Vinnytsia_commercial.csv", "Zaporizhzhia_commercial.csv",
    "Zhytomyr_commercial.csv"
]

dfs = []
for filename in files:
    df = pd.read_csv(filename)
    dfs.append(df)
    print(f"✓ {filename}")

df_ukraine = pd.concat(dfs, ignore_index=True)

df_kyiv = pd.read_csv("kyiv_commercial.csv")

✓ Cherkasy_commercial.csv
✓ Chernihiv_commercial.csv
✓ Chernivtsi_commercial.csv
✓ Dnipro_commercial.csv
✓ Ivano-Frankivsk_commercial.csv
✓ Kharkiv_commercial.csv
✓ Khmelnytskyi_commercial.csv
✓ Kropyvnytskyi_commercial.csv
✓ Lutsk_commercial.csv
✓ Lviv_commercial.csv
✓ Mykolaiv_commercial.csv
✓ Odesa_commercial.csv
✓ Poltava_commercial.csv
✓ Rivne_commercial.csv
✓ Sumy_commercial.csv
✓ Ternopil_commercial.csv
✓ Uzhhorod_commercial.csv
✓ Vinnytsia_commercial.csv
✓ Zaporizhzhia_commercial.csv
✓ Zhytomyr_commercial.csv


In [2]:
df_ukraine.isna().sum()

group_id                     0
has_duplicates               0
url                          0
city                         0
district                  1648
residential_complex       1075
lat                          0
lon                          0
distance_to_center_km        0
poi_name                   355
poi_distance_m             355
geo_region                   0
price                        0
area                         0
floor                     1638
floor_count                594
house_type                 752
wall_type                  838
heating                    798
ceiling_height             859
year_of_building           660
is_bank                      0
is_office                    0
is_services                  0
is_warehouse                 0
is_production                0
is_free                      0
is_retail                    0
is_garage                    0
is_parking_spot              0
has_tenant                   0
rental_price              1639
implied_

In [3]:
df_kyiv.isna().sum()

group_id                     0
has_duplicates               0
url                          0
city                         0
district                     0
residential_complex       1512
lat                          0
lon                          0
distance_to_center_km        0
poi_name                  1892
poi_distance_m            1892
geo_region                   0
price                        0
area                         0
floor                     2749
floor_count                261
house_type                 265
wall_type                  261
heating                    256
ceiling_height             262
year_of_building           226
is_bank                      0
is_office                    0
is_services                  0
is_warehouse                 0
is_production                0
is_free                      0
is_retail                    0
is_garage                    0
is_parking_spot              0
has_tenant                   0
rental_price              2707
implied_

In [4]:
before = len(df_ukraine)
df_ukraine = df_ukraine[df_ukraine["price"].notna()]
df_ukraine = df_ukraine[df_ukraine["price"] > 10]
print(f"Прибрано {before - len(df_ukraine)} рядків (пропущена/аномальна ціна). Лишилось: {len(df_ukraine)}")

Прибрано 0 рядків (пропущена/аномальна ціна). Лишилось: 1648


In [5]:
before = len(df_kyiv)
df_kyiv = df_kyiv[df_kyiv["price"].notna()]
df_kyiv = df_kyiv[df_kyiv["price"] > 10]
print(f"Прибрано {before - len(df_kyiv)} рядків (пропущена/аномальна ціна). Лишилось: {len(df_kyiv)}")

Прибрано 0 рядків (пропущена/аномальна ціна). Лишилось: 2755


In [6]:
def apply_price_sqm_sanity_filter(df: pd.DataFrame, low: float = 5.0, high: float = 20000.0) -> pd.DataFrame:
    ratio = df["price"] / df["area"].replace(0, np.nan)
    mask = ratio.between(low, high)
    dropped = (~mask).sum()
    if dropped:
        print(f"  Прибрано {dropped} рядків з неправдоподібним $/м² (поза [{low}, {high}])")
    return df[mask]


df_ukraine = apply_price_sqm_sanity_filter(df_ukraine)
df_kyiv = apply_price_sqm_sanity_filter(df_kyiv)

  Прибрано 1 рядків з неправдоподібним $/м² (поза [5.0, 20000.0])
  Прибрано 16 рядків з неправдоподібним $/м² (поза [5.0, 20000.0])


In [7]:
import numpy as np


def object_group(df: pd.DataFrame) -> pd.Series:
    is_parking = df["is_garage"].astype(bool) | df["is_parking_spot"].astype(bool)
    is_warehouse = df["is_warehouse"].astype(bool)
    return np.select(
        [is_parking, is_warehouse],
        ["parking_or_garage", "warehouse"],
        default="premises",
    )


def remove_outliers_iqr_grouped(df: pd.DataFrame, cols: list[str], k: float = 1.5) -> pd.DataFrame:
    groups = object_group(df)
    keep_mask = pd.Series(True, index=df.index)

    for group_value in pd.unique(groups):
        idx = df.index[groups == group_value]
        group_df = df.loc[idx]
        group_mask = pd.Series(True, index=idx)
        for col in cols:
            q1, q3 = group_df[col].quantile(0.25), group_df[col].quantile(0.75)
            iqr = q3 - q1
            lower, upper = q1 - k * iqr, q3 + k * iqr
            group_mask &= group_df[col].between(lower, upper)
            print(f"  [{group_value}] {col}: lower={lower:.1f}, upper={upper:.1f}")
        keep_mask.loc[idx] = group_mask

    return df[keep_mask]


In [8]:
before = len(df_ukraine)
df_ukraine = remove_outliers_iqr_grouped(df_ukraine, ["price", "area"])
print(f"\nПрибрано {before - len(df_ukraine)} викидів по Україні. Лишилось: {len(df_ukraine)}")

  [parking_or_garage] price: lower=-4687.5, upper=35812.5
  [parking_or_garage] area: lower=5.0, upper=33.8
  [warehouse] price: lower=-300000.0, upper=644000.0
  [warehouse] area: lower=-1017.5, upper=1890.5
  [premises] price: lower=-181125.0, upper=461875.0
  [premises] area: lower=-109.9, upper=316.7

Прибрано 226 викидів по Україні. Лишилось: 1421


In [9]:
before_kyiv = len(df_kyiv)
df_kyiv = remove_outliers_iqr_grouped(df_kyiv, ["price", "area"])
print(f"\nПрибрано {before_kyiv - len(df_kyiv)} викидів в Києві. Лишилось: {len(df_kyiv)}")

  [warehouse] price: lower=-874850.0, upper=1724750.0
  [warehouse] area: lower=-635.0, upper=1261.0
  [premises] price: lower=-410000.0, upper=1030000.0
  [premises] area: lower=-144.5, upper=427.5
  [parking_or_garage] price: lower=-5000.0, upper=75000.0
  [parking_or_garage] area: lower=0.6, upper=36.4

Прибрано 394 викидів в Києві. Лишилось: 2345


In [10]:
for df in (df_ukraine, df_kyiv):
    df["in_residential_complex"] = df["residential_complex"].notna().astype(int)
    df["has_poi"] = df["poi_name"].notna().astype(int)
    df.drop(columns=["residential_complex", "poi_name"], inplace=True)

In [11]:
for df in (df_ukraine, df_kyiv):
    df.drop(columns=["floor"], inplace=True)

In [12]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

NUMERIC_TARGETS = ["year_of_building", "ceiling_height"]
PURPOSE_COLS = [
    "is_bank", "is_office", "is_services", "is_warehouse",
    "is_production", "is_free", "is_retail", "is_garage", "is_parking_spot",
]


def impute_premises_numeric(df: pd.DataFrame, location_cols: list[str]) -> pd.DataFrame:
    premises_mask = (object_group(df) == "premises")
    idx = df.index[premises_mask]
    subset = df.loc[idx]

    location_onehot = pd.get_dummies(subset[location_cols], columns=location_cols)

    numeric_cols = ["area"] + NUMERIC_TARGETS
    scaler = StandardScaler()
    scaled_numeric = pd.DataFrame(
        scaler.fit_transform(subset[numeric_cols]),
        columns=numeric_cols, index=idx,
    )

    knn_input = pd.concat(
        [scaled_numeric,
         subset[PURPOSE_COLS].reset_index(drop=True).set_axis(idx),
         location_onehot.reset_index(drop=True).set_axis(idx)],
        axis=1,
    )

    imputer = KNNImputer(n_neighbors=5, weights="distance")
    imputed_array = imputer.fit_transform(knn_input)
    imputed = pd.DataFrame(imputed_array, columns=knn_input.columns, index=idx)

    imputed_numeric = pd.DataFrame(
        scaler.inverse_transform(imputed[numeric_cols]),
        columns=numeric_cols, index=idx,
    )

    df.loc[idx, "year_of_building"] = imputed_numeric["year_of_building"].round().astype("Int64")
    df.loc[idx, "ceiling_height"] = imputed_numeric["ceiling_height"].round(1)

    print(f"  Імпутовано {len(idx)} рядків типу 'premises' "
          f"({premises_mask.sum()}/{len(df)} загалом; решта — гараж/паркомісце/склад, лишені NaN)")
    return df


In [13]:
df_ukraine = impute_premises_numeric(df_ukraine, location_cols=["district", "city"])
df_ukraine[NUMERIC_TARGETS].isna().sum()

  Імпутовано 465 рядків типу 'premises' (465/1421 загалом; решта — гараж/паркомісце/склад, лишені NaN)


year_of_building    528
ceiling_height      706
dtype: int64

In [14]:
df_ukraine = impute_premises_numeric(df_ukraine, location_cols=["district", "city"])
df_ukraine[NUMERIC_TARGETS].isna().sum()

  Імпутовано 465 рядків типу 'premises' (465/1421 загалом; решта — гараж/паркомісце/склад, лишені NaN)


year_of_building    528
ceiling_height      706
dtype: int64

In [15]:
for df in (df_ukraine, df_kyiv):
    for col in ["house_type", "wall_type", "heating"]:
        df[col] = df[col].fillna("Unknown")

df_ukraine[["house_type", "wall_type", "heating"]].isna().sum()

house_type    0
wall_type     0
heating       0
dtype: int64

In [16]:
df_kyiv[["house_type", "wall_type", "heating"]].isna().sum()

house_type    0
wall_type     0
heating       0
dtype: int64

In [17]:
df_ukraine["price"] = df_ukraine["price"].round(0)
df_ukraine["area"] = df_ukraine["area"].round(1)
df_ukraine[["price", "area", "ceiling_height"]].describe()

,price,area,ceiling_height
count,1421.000000,1421.000000,715.000000
mean,111047.867699,188.558269,2.838769
std,114841.923644,312.330176,0.266515
min,700.000000,1.000000,2.400000
25%,25000.000000,30.000000,2.700000
50%,70000.000000,78.200000,2.800000
75%,155000.000000,170.000000,3.000000
max,630000.000000,1800.000000,4.000000


In [18]:
df_kyiv["price"] = df_kyiv["price"].round(0)
df_kyiv["area"] = df_kyiv["area"].round(1)
df_kyiv[["price", "area", "ceiling_height"]].describe()

,price,area,ceiling_height
count,2.345000e+03,2345.000000,2178.000000
mean,2.368205e+05,120.200128,2.918057
std,2.246491e+05,118.816034,0.358032
min,2.000000e+03,2.200000,2.000000
25%,7.650000e+04,46.000000,2.700000
50%,1.680000e+05,88.000000,2.850000
75%,3.300000e+05,155.000000,3.000000
max,1.650000e+06,1234.000000,5.000000


In [19]:
df_ukraine[df_ukraine["area"] == 1]

,group_id,has_duplicates,url,city,district,lat,lon,distance_to_center_km,poi_distance_m,geo_region,...,has_generator,has_autonomous_heating,has_gas_boiler,separate_entrance,basement_level,without_commission,is_exclusive,text,in_residential_complex,has_poi
1102,101563953377627182,False,https://lun.ua/realty/4235975105,Lviv,NaN,49.811455,24.057589,3.723757,1465.0,West,...,0,0,0,0,0,0,0,"Продаж комірки в ЖК Аурум Спарк, вул. Навроцьк...",1,1


In [20]:
df_kyiv.to_csv("Kyiv_commercial_for_analysis.csv", index=False)
df_ukraine.to_csv("Ukraine_commercial_for_analysis.csv", index=False)

In [21]:
before = len(df_ukraine)
df_encoded_ukraine = df_ukraine.drop_duplicates(subset="group_id", keep="first").copy()
print(f"Прибрано {before - len(df_encoded_ukraine)} дублікатів за group_id (Україна)")

df_encoded_ukraine = df_encoded_ukraine.drop(columns=["url", "group_id", "geo_region", "text", "lat", "lon", "has_duplicates"])

categorical_cols_ukraine = ["house_type", "wall_type", "heating", "city"]
df_encoded_ukraine = df_encoded_ukraine.drop(columns=["district"])

df_encoded_ukraine = pd.get_dummies(df_encoded_ukraine, columns=categorical_cols_ukraine, drop_first=True)
df_encoded_ukraine.to_csv("Ukraine_commercial_ML.csv", index=False)
df_encoded_ukraine.shape

Прибрано 4 дублікатів за group_id (Україна)


(1417, 69)

In [22]:
before = len(df_kyiv)
df_encoded_kyiv = df_kyiv.drop_duplicates(subset="group_id", keep="first").copy()
print(f"Прибрано {before - len(df_encoded_kyiv)} дублікатів за group_id (Київ)")

df_encoded_kyiv = df_encoded_kyiv.drop(columns=["url", "group_id", "geo_region", "text", "lat", "lon", "has_duplicates"])

df_encoded_kyiv = df_encoded_kyiv.drop(columns=["city"])
categorical_cols_kyiv = ["house_type", "wall_type", "heating", "district"]

df_encoded_kyiv = pd.get_dummies(df_encoded_kyiv, columns=categorical_cols_kyiv, drop_first=True)
df_encoded_kyiv.to_csv("Kyiv_commercial_ML.csv", index=False)
df_encoded_kyiv.shape

Прибрано 103 дублікатів за group_id (Київ)


(2242, 62)